# When it was built, and how it was sold

`YearBuilt` runs from 1872 to 2010. `YearRemodAdd` runs from 1950 to 2010. On 764 rows the two years are equal. On the other $1460 - 764 = 696$ rows the remodel year is later.

Age at sale is `YrSold - YearBuilt`. It runs from 0 to 136 years. The sales themselves fall in five years:

| Year sold | Houses |
|---:|---:|
| 2006 | 314 |
| 2007 | 329 |
| 2008 | 304 |
| 2009 | 338 |
| 2010 | 175 |

Those five counts add to 1460. The busiest month is June, with 253 sales. February has 52.

`SaleCondition` says what kind of bargain it was. `Normal` is 1198 of the sales. `Partial` is 125: the house was not finished at the last assessment, which is common for a new home. `Abnorml` is 101. The rest are small: family sales, allocation, and adjoining land.

A new house and a partial sale are easy to confuse. House 1 was built in 2003 and sold in 2008 under `Normal`, five years old. The partial sales include houses that are both new and unusually large. Those rows are the next lesson.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "14-House Prices" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "data_description.txt").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["Id"], train[0]["SalePrice"])


1460 1459 1459
1 208500


In [3]:
# Years, age, month, and sale condition.
from collections import Counter
built = [int(row["YearBuilt"]) for row in train]
remod = [int(row["YearRemodAdd"]) for row in train]
ages = [int(row["YrSold"]) - int(row["YearBuilt"]) for row in train]
print(min(built), max(built), min(remod), max(remod))
print(sum(b == r for b, r in zip(built, remod)), min(ages), max(ages))
years = Counter(row["YrSold"] for row in train)
months = Counter(int(row["MoSold"]) for row in train)
conditions = Counter(row["SaleCondition"] for row in train)
print(dict(years))
print(months[6], months[2])
print(dict(conditions))
assert (min(built), max(built)) == (1872, 2010)
assert (min(remod), max(remod)) == (1950, 2010)
assert sum(b == r for b, r in zip(built, remod)) == 764
assert (min(ages), max(ages)) == (0, 136)
assert years == {"2006": 314, "2007": 329, "2008": 304, "2009": 338, "2010": 175}
assert sum(years.values()) == 1460
assert months[6] == 253 and months[2] == 52
assert conditions["Normal"] == 1198
assert conditions["Partial"] == 125
assert conditions["Abnorml"] == 101


1872 2010 1950 2010
764 0 136
{'2008': 304, '2007': 329, '2006': 314, '2009': 338, '2010': 175}
253 52
{'Normal': 1198, 'Abnorml': 101, 'Partial': 125, 'AdjLand': 4, 'Alloca': 12, 'Family': 20}


## Pitfall

2010 contains 175 sales, not a partial year you should drop, and it is smaller than the other four years. A model that treats the year number as a strong price trend is leaning on five nearby years. The construction year spans 1872 to 2010 and moves much more.

## Summary

Construction runs from 1872 to 2010. Remodel matches construction on 764 rows. Sales run from 2006 to 2010, and 1198 of them are `Normal`.
